# PCA: the best view of your data
**Machine Learning · Week 6 (Learning without labels) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/ml/w6d3.html

**PCA** (principal component analysis) rotates the axes to point along the directions where the data
varies most, then keeps only the top few. Fewer numbers per data point, as little information lost as possible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)
cloud = rng.multivariate_normal([0, 0], [[3.0, 2.2], [2.2, 2.2]], size=300)
pca = PCA(n_components=2).fit(cloud)
print("principal directions (rows):\n", pca.components_.round(3))
print("share of the variance      :", pca.explained_variance_ratio_.round(3))

fig, ax = plt.subplots(figsize=(4.8, 4.2), layout="constrained")
ax.scatter(cloud[:, 0], cloud[:, 1], s=10, color="#b9a7ff")
for direction, var, color in zip(pca.components_, pca.explained_variance_, ["#ff76a8", "#17a98c"]):
    ax.annotate("", xy=2 * np.sqrt(var) * direction, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", lw=2.5, color=color))
ax.set_aspect("equal")
ax.set_title("PC1 (pink) and PC2 (green)")
plt.show()

## PCA by hand
Center the data, compute the covariance matrix, and take its eigenvectors: those are the principal
directions, and the eigenvalues are the variances along them.

In [ ]:
centered = cloud - cloud.mean(axis=0)
cov = centered.T @ centered / (len(cloud) - 1)
values, vectors = np.linalg.eigh(cov)                 # eigh: for symmetric matrices, ascending order
order = np.argsort(values)[::-1]                      # largest variance first
values, vectors = values[order], vectors[:, order]
print("eigenvalues (variances):", values.round(3), "  scikit-learn:", pca.explained_variance_.round(3))
print("PC1 by hand:", vectors[:, 0].round(3), "  scikit-learn:", pca.components_[0].round(3), " (the sign may flip)")

# keep only PC1: what is kept, what is lost?
z = centered @ vectors[:, 0]                          # one number per point
rebuilt = np.outer(z, vectors[:, 0])                  # back in 2D, on the PC1 line
lost = ((centered - rebuilt) ** 2).sum(axis=1).mean()
print(f"variance kept {z.var(ddof=1):.3f} + squared error lost {lost * len(cloud) / (len(cloud) - 1):.3f} "
      f"= total {np.trace(cov):.3f}")

Keeping the most variance and losing the least squared error are the same goal: they always add up
to the total.

## 64 pixels down to 2 numbers

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data / 16.0                     # 1797 images, 64 pixel values in [0, 1]
pca2 = PCA(n_components=2).fit(X)
Z = pca2.transform(X)
print("variance kept by 2 of 64 components:", round(pca2.explained_variance_ratio_.sum(), 3))

fig, ax = plt.subplots(figsize=(5.4, 4.2), layout="constrained")
points = ax.scatter(Z[:, 0], Z[:, 1], c=digits.target, cmap="tab10", s=7)
fig.colorbar(points, ax=ax, label="digit", ticks=range(10))
ax.set_title("Digits after PCA to 2D (labels only for color)")
plt.show()

## How many components? The explained-variance curve

In [ ]:
full = PCA().fit(X)
cumulative = np.cumsum(full.explained_variance_ratio_)
k90 = int(np.searchsorted(cumulative, 0.90)) + 1
plt.plot(np.arange(1, 65), cumulative, color="#7b5cfa", lw=2.5)
plt.axhline(0.9, color="#17a98c", ls="--", lw=1.5)
plt.axvline(k90, color="#ff8a50", ls="--", lw=1.5)
plt.xlabel("number of components kept")
plt.ylabel("share of the variance kept")
plt.title(f"90% of the variance with {k90} of 64 components")
plt.show()

fig, axes = plt.subplots(1, 6, figsize=(8, 1.8), layout="constrained")
for ax, k in zip(axes, [1, 2, 5, 10, 20, 64]):
    p = PCA(n_components=k).fit(X)
    ax.imshow(p.inverse_transform(p.transform(X[:1]))[0].reshape(8, 8), cmap="Purples", vmin=0, vmax=1)
    ax.set_title(f"k = {k}")
    ax.axis("off")
plt.show()

## Your turn: PCA before a classifier
Squeeze the digits to $k$ components, then train a logistic regression on them. For
$k = 2, 5, 10, 20, 40, 64$, report the test accuracy. How small can $k$ be before accuracy suffers?
(Fit the PCA on the training set only: the test set stays unseen.)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(X, digits.target, test_size=0.25, random_state=0, stratify=digits.target)
for k in [2, 5, 10, 20, 40, 64]:
    model = None   # TODO: make_pipeline(PCA(n_components=k), LogisticRegression(max_iter=2000)), fit on the training set
    # TODO: print k and the test accuracy

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(X, digits.target, test_size=0.25, random_state=0, stratify=digits.target)
for k in [2, 5, 10, 20, 40, 64]:
    model = make_pipeline(PCA(n_components=k), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
    print(f"k = {k:2d}: test accuracy {model.score(X_test, y_test):.3f}")
print("20 components get within about 1% of all 64, and 40 match them: the last directions add almost nothing.")